# Forecasting Baselines: Short-Term Point Forecasts for Estimated WEC Power

This notebook builds simple short-term point-forecasting baselines for the estimated WEC power output prepared in the previous notebooks.

The input data used here is the 30-minute Leixões estimated WEC power time series. The sourcing and sea-state preparation are documented in [wave data preparation](01_wave_data_preparation.ipynb), and the simplified WEC power estimation procedure is documented in [WEC power estimation](02_wec_power_estimation.ipynb).

The main modelling target is `wec_power_norm_estimated`, which represents the estimated WEC power normalized by the nominal 250 kW rating used for the generic WEC proxy. This device-relative scale is useful for modelling because it keeps the target independent of larger rated-power or array-sizing assumptions.

For physical interpretation in plots and reported errors, the normalized target is also shown on the corresponding 250 kW scale:

`wec_power_kw_estimated = wec_power_norm_estimated × 250 kW`

The forecast target is not measured WEC power. It is also not a validated device model, hydrodynamic simulation, or wave-to-wire model. It is a simplified, transparent WEC power proxy derived from observed sea-state variables using the assumptions documented in the WEC power estimation notebook.

The purpose of this notebook is to create reproducible point-forecast baselines and residuals that can support the uncertainty and storage-aware subsequent analyses.

The benchmark uses a compact power-only autoregressive feature set. Wave-height, wave-period, and wave-power-flux variables are not used as predictors in the main benchmark, because the estimated WEC power target was itself constructed from sea-state assumptions. This avoids a forecast model that partly relearns the constructed power-matrix mapping instead of forecasting the estimated power series as a time-dependent signal.

The feature set is intentionally kept small: selected lag values, rolling means, and rolling standard deviations of the estimated power series. This keeps the benchmark interpretable and avoids turning the first forecasting notebook into a broad feature-search exercise.

The forecasting horizons are:

| Horizon | Steps | Operational context |
|---:|---:|---|
| 30 min | 1 | Grid-relevant: next balancing or dispatch interval |
| 1 h | 2 | Grid/storage-relevant: near-term operational planning |
| 2 h | 4 | Storage-relevant: short storage dispatch window |
| 4 h | 8 | Grid/storage-relevant: upper short-term benchmark without external wave forecast inputs |

Missing target values are not imputed. Lag, rolling, and future target values are created only within continuous valid segments of the original 30-minute time axis. This prevents observations separated by data gaps from being treated as adjacent 30-minute observations.

Evaluation uses chronological rolling-origin folds. Each fold contains a training block, a calibration/validation block, and a final test block. Using several rolling-origin test folds gives multiple performance estimates across different time periods, which helps assess the stability of the forecasting baselines rather than relying on a single final split.

The notebook compares a small set of interpretable baseline models:

- Persistence
- Rolling mean
- Ridge regression
- Random forest regression

Performance is reported using MAE, RMSE, and skill scores relative to persistence.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True


# Paths
data_path = Path("../data/processed/leixoes_wec_power_30min_estimated.parquet")

output_dir = Path("../outputs/notebook_03")
metrics_path = output_dir / "forecast_metrics.csv"
predictions_path = output_dir / "forecast_predictions.parquet"
folds_path = output_dir / "forecast_folds.csv"


# Target and physical interpretation scale
time_col = "time"
target_col = "wec_power_norm_estimated"
rated_power_kw = 250.0


# Forecast horizons for 30-minute data
horizons = {
    1: {
        "horizon_label": "30 min",
        "horizon_hours": 0.5,
        "operational_context": "Grid-relevant: next balancing or dispatch interval",
    },
    2: {
        "horizon_label": "1 h",
        "horizon_hours": 1.0,
        "operational_context": "Grid/storage-relevant: near-term operational planning",
    },
    4: {
        "horizon_label": "2 h",
        "horizon_hours": 2.0,
        "operational_context": "Storage-relevant: short storage dispatch window",
    },
    8: {
        "horizon_label": "4 h",
        "horizon_hours": 4.0,
        "operational_context": "Grid/storage-relevant: upper short-term benchmark without external wave forecast inputs",
    },
}


# Power-only autoregressive features
lag_steps = [1, 2, 4, 8, 24]
roll_mean_windows = [3, 6, 12]
roll_std_windows = [6, 12]

feature_cols = (
    [f"power_lag_{step}" for step in lag_steps]
    + [f"power_roll_mean_{window}" for window in roll_mean_windows]
    + [f"power_roll_std_{window}" for window in roll_std_windows]
)


# Reproducibility
random_state = 42